# Export results and checkpoints for the public release  (CPU runtime, ~10–40 min)

Run in the Google account whose Drive holds `MyDrive/SSL_AgriStudy/`. Nothing on Drive is modified. Produces:

1. **`ssl-agri-scale-results.zip`**: copy its contents into the repository's `results/` folder and commit.
   * `splits/` (carved validation lists), every result JSON and `*_runs/` folder, `correct/` (per-image
     correctness for every backbone × dataset × protocol, needed for every paired test), `paper_tables/` and
     `paper_figures/`, PlantDoc `SPLIT_INFO.json`
2. **`checkpoints_for_zenodo/`**: for every finished laboratory-scale run, the **teacher backbone only** (the
   weights all evaluations use; ~87 MB instead of ~700 MB), named `<run>_teacher_backbone.pth`, plus
   `checkpoint_manifest.csv` with SHA-256, size, final epoch and the training arguments. Upload the folder to
   Zenodo (or the Hugging Face Hub) and copy the manifest into `results/`.

In [ ]:
import os, sys, json, shutil, hashlib, glob, zipfile, time
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
import torch, pandas as pd
BASE = os.environ.get('SSL_BASE', '/content/drive/MyDrive/SSL_AgriStudy')
OUT = os.environ.get('SSL_EXPORT', '/content/export')
RES = f'{BASE}/results_v2'
EXPORT_CHECKPOINTS = True        # set False to export only the results
shutil.rmtree(OUT, ignore_errors=True); os.makedirs(f'{OUT}/results', exist_ok=True)
assert os.path.isdir(RES), f'{RES} not found — wrong account or folder?'
print('source:', BASE)

In [ ]:
# ── 1. Results ──
R = f'{OUT}/results'
for f in sorted(glob.glob(f'{RES}/*.json')):
    shutil.copy(f, R)
for d in sorted(glob.glob(f'{RES}/*_runs')) + [f'{RES}/splits', f'{RES}/correct']:
    if os.path.isdir(d):
        shutil.copytree(d, f'{R}/{os.path.basename(d)}')
for src, dst in [(f'{BASE}/paper_v2/tables', 'paper_tables'), (f'{BASE}/paper_v2/figures', 'paper_figures')]:
    if os.path.isdir(src):
        shutil.copytree(src, f'{R}/{dst}')
pd_info = f'{BASE}/datasets/PlantDoc/SPLIT_INFO.json'
if os.path.exists(pd_info):
    os.makedirs(f'{R}/splits', exist_ok=True); shutil.copy(pd_info, f'{R}/splits/PlantDoc_SPLIT_INFO.json')
n = sum(len(fs) for _, _, fs in os.walk(R)); mb = sum(os.path.getsize(os.path.join(p, f)) for p, _, fs in os.walk(R) for f in fs) / 1e6
print(f'{n} files, {mb:.1f} MB')
zp = f'{OUT}/ssl-agri-scale-results.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for p, _, fs in os.walk(R):
        for f in fs:
            z.write(os.path.join(p, f), os.path.relpath(os.path.join(p, f), R))
print('✓', zp, f'{os.path.getsize(zp) / 1e6:.1f} MB')

In [ ]:
# ── 2. Teacher backbones + manifest for Zenodo ──
def sha256(path, buf=1 << 24):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(buf), b''): h.update(b)
    return h.hexdigest()
runs = json.load(open(f'{RES}/08_10_pretraining.json')) if os.path.exists(f'{RES}/08_10_pretraining.json') else {}
for f in glob.glob(f'{RES}/08_10_pretraining_runs/*.json'):
    try:
        rec = json.load(open(f)); runs[rec['__key']] = rec['record']
    except Exception:
        pass
rows = []
if EXPORT_CHECKPOINTS:
    CK = f'{OUT}/checkpoints_for_zenodo'; os.makedirs(CK, exist_ok=True)
    for sub in ('domain_dino', 'domain_dino_v2'):
        for d in sorted(glob.glob(f'{BASE}/checkpoints/{sub}/*')):
            name, ck = os.path.basename(d), f'{d}/checkpoint.pth'
            complete = os.path.exists(f'{d}/COMPLETE') or sub == 'domain_dino'
            if not os.path.exists(ck) or not complete:
                print(f'skip {name}: ' + ('no checkpoint' if not os.path.exists(ck) else 'not COMPLETE')); continue
            t0 = time.time()
            sd = torch.load(ck, map_location='cpu', weights_only=False)
            ep = sd.get('epoch') if isinstance(sd, dict) else None
            teacher = sd['teacher'] if isinstance(sd, dict) and 'teacher' in sd else sd
            bb = {k.replace('module.', '').replace('backbone.', ''): v for k, v in teacher.items()
                  if not k.replace('module.', '').replace('backbone.', '').startswith('head')}
            assert len(bb) == 150, f'{name}: expected 150 ViT-S/16 backbone tensors, got {len(bb)}'
            out = f'{CK}/{name}_teacher_backbone.pth'
            torch.save(bb, out)
            r = runs.get(name, {})
            rows.append({'run': name, 'file': os.path.basename(out), 'sha256': sha256(out),
                         'size_MB': round(os.path.getsize(out) / 1e6, 1), 'final_epoch': ep,
                         'source_checkpoint_sha256': sha256(ck) if os.environ.get('SSL_HASH_SOURCE') else '',
                         'n_images': r.get('n_images'), 'steps': r.get('steps'), 'init': r.get('init'),
                         'lr': r.get('lr'), 'epochs': r.get('epochs'), 'seed': r.get('seed'),
                         'extra_args': ' '.join(r.get('extra_args', []))})
            print(f'✓ {name:28s} epoch {ep}  {rows[-1]["size_MB"]} MB  ({time.time() - t0:.0f}s)')
    if not rows:
        raise RuntimeError('No finished checkpoints found under checkpoints/domain_dino*/ — wrong Drive account?')
    M = pd.DataFrame(rows); M.to_csv(f'{CK}/checkpoint_manifest.csv', index=False)
    M.to_csv(f'{OUT}/results/checkpoint_manifest.csv', index=False)
    print(f'\n{len(M)} backbones, {M.size_MB.sum() / 1e3:.2f} GB in {CK}')

In [ ]:
# ── 3. Save the export to Drive (a NEW folder; nothing existing is touched) and finish ──
if IN_COLAB:
    dst = f'/content/drive/MyDrive/ssl-agri-scale-release-{time.strftime("%Y%m%d")}'
    shutil.copytree(OUT, dst, dirs_exist_ok=True)
    print('copied to', dst)
    from google.colab import drive
    drive.flush_and_unmount()
    print('✓ All files uploaded to Drive. You can close this runtime now.')